# SpikeVPR — tutorial

A short walkthrough of the release:

1. Load a pretrained SpikeVPR model and run it on an event frame.
2. Build one of the three datasets and measure recall@N.
3. Estimate the model's inference energy from its measured spike rate.
4. Compare against the NetVLAD (ANN) baseline.

Run this notebook from the `src/` directory (so `weights/` and `configs/` resolve).
Everything here runs on CPU with small subsets.

In [ ]:
import os, sys, yaml, torch
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')        # move to src/
sys.path.insert(0, os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

def load_config(path):
    with open(path) as f:
        return yaml.safe_load(f)

## 1. Load a pretrained model

`build_spikevpr` returns the SEW-ResNet backbone + MixVPR head. The descriptor is
4096-D and L2-normalised. Use `neuron_type='LIFNode'` for the Brisbane/NSAVP
checkpoints and `'IFNode'` for NYC (this matches how each was trained).

In [ ]:
from spikevpr.models import build_spikevpr, count_parameters

model = build_spikevpr('sew_resnet34',
                       checkpoint='weights/sew_resnet34_nsavp.pth',
                       neuron_type='LIFNode', device=device, eval_mode=True)
print(f'{count_parameters(model)/1e6:.1f}M trainable params')

frame = torch.zeros(1, 2, 260, 346)            # one ON/OFF event frame
with torch.no_grad():
    descriptor = model(frame.to(device))
print('descriptor:', tuple(descriptor.shape))  # (1, 4096)

## 2. Build a dataset and measure recall@N

`build_datasets` returns the train/eval datasets for a dataset name. For Brisbane
we shrink it with `n_places` so the notebook is fast; drop that for the full set.
Make sure `configs/brisbane.yaml` points at your local Brisbane copy first.

In [ ]:
from spikevpr.data.loaders import build_datasets, make_loader
from spikevpr.evaluation.metrics import extract_pair_embeddings, similarity_matrix, recall_at_n

cfg = load_config('configs/brisbane.yaml')
cfg['data']['n_places'] = 60                    # subset for a quick run

bris = build_spikevpr('sew_resnet18', checkpoint='weights/sew_resnet18_brisbane.pth',
                      neuron_type='LIFNode', device=device, eval_mode=True)
datasets = build_datasets('brisbane', cfg)
loader = make_loader(datasets['eval'], batch_size=8, shuffle=False, num_workers=0)

q_emb, q_gps, r_emb, r_gps = extract_pair_embeddings(bris, loader, device)
sim = similarity_matrix(q_emb, r_emb)
print(recall_at_n(sim, q_gps, r_gps, threshold=30, n_values=(1, 5, 10)))

The same `evaluate(...)` call works for every dataset (NYC uses cross-session
strict recall internally):

```python
from spikevpr.evaluation import evaluate
evaluate('nsavp', load_config('configs/nsavp.yaml'), model, device)
```

## 3. Estimate inference energy

`estimate_snn_energy` measures the average spike rate over a few batches of real
data and applies the Dampfhoffer / Lemaire 45 nm proxies to the convolutional
backbone. Nothing is hardcoded — every value is recomputed from the model.

In [ ]:
from torch.utils.data import Subset
from spikevpr.energy.estimate import estimate_snn_energy

ncfg = load_config('configs/nsavp.yaml')
nsavp = build_datasets('nsavp', ncfg)
eloader = make_loader(Subset(nsavp['eval'], list(range(16))), 8, shuffle=False, num_workers=0)

res = estimate_snn_energy(model, eloader, device, num_batches=2)
print(f"spikes/synapse : {res['avg_spikes_per_syn']*100:.4f}%")
print(f"Dampfhoffer    : {res['methods']['dampfhoffer']/1e9:.2f} mJ")
print(f"Lemaire        : {res['methods']['lemaire']/1e9:.2f} mJ")

## 4. Compare against NetVLAD (ANN)

The NetVLAD event-VPR baseline (EST + ResNet-34 + NetVLAD + WPCA) is the ANN
reference. Its energy is recomputed the same way (via measured ReLU sparsity).
Run the full comparison from the CLI:

```bash
python -m spikevpr.energy.compare --dataset nsavp --config configs/nsavp.yaml \
    --encoder sew_resnet34 --checkpoint weights/sew_resnet34_nsavp.pth \
    --netvlad weights/netvlad_weights.pth --wpca weights/wpca_weights.pth
```

A minimal in-notebook version of the ANN side:

In [ ]:
from spikevpr.baselines.netvlad import RetrievalModel
from spikevpr.energy.estimate import (extract_layers, measure_relu_sparsity,
                                      assign_relu_sparsity_to_layers, estimate_model)
from spikevpr.energy.compare import _synthetic_events, _EventBatches

netvlad = RetrievalModel.from_weights('weights/netvlad_weights.pth',
                                      'weights/wpca_weights.pth',
                                      device=device, voxel_dimension=(9, 260, 346)).eval()
events = _synthetic_events(n_events=3000, device=device)
layers = extract_layers(netvlad, events)
sparsity = measure_relu_sparsity(netvlad, _EventBatches(3, device), device, num_batches=3)
layers = assign_relu_sparsity_to_layers(layers, sparsity)
ann = estimate_model('NetVLAD+ResNet34', layers, mode='ann')
print(f"NetVLAD Lemaire energy: {ann['methods']['lemaire']/1e9:.1f} mJ "
      f"(SpikeVPR is ~{ann['methods']['lemaire']/res['methods']['lemaire']:.0f}x lower)")

## Next steps

- Train from scratch: `python -m spikevpr.training.train --dataset brisbane --config configs/brisbane.yaml --encoder sew_resnet34 --output_folder runs/brisbane_r34`
- Swap datasets by changing the `--dataset` / `--config` pair (`brisbane`, `nsavp`, `nyc`).
- See `CHANGES.md` for how this package relates to the original research code, and `DATASETS.md` for dataset layout.